# **Exception Handling in OOP**

**DS212 — Object-Oriented Programming**

**Week 11:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Design robust classes using exception handling, custom exceptions, and defensive programming.

Real programs meet **unexpected situations**: missing files, invalid input, network failures, impossible values in a dataset. A robust program does not crash with a confusing message — it **detects** the problem, **reports** it clearly, and **recovers** when it can.

Python handles these situations with **exceptions**. And because Python is object-oriented, **exceptions are objects**, organized in a **class hierarchy** that we can extend with our own exception classes.

```{mermaid}
flowchart LR
    A["Something goes wrong"] --> B["An exception object<br/>is raised"]
    B --> C{"Is there a matching<br/>except block?"}
    C -- yes --> D["Handle it<br/>program continues"]
    C -- no --> E["Program stops<br/>traceback shown"]
```

## **Errors and Exceptions**

**Syntax errors** are detected *before* the program runs — the code is not valid Python:

```python
if score > 75        # SyntaxError: expected ':'
    print("pass")
```

**Exceptions** happen *while* the program runs, even when the syntax is correct:

In [ ]:
examples = [
    lambda: 10 / 0,
    lambda: int("eighty"),
    lambda: [1, 2, 3][10],
    lambda: {"name": "Ana"}["age"],
    lambda: "DS" + 212,
    lambda: None.upper(),
]

for run in examples:
    try:
        run()
    except Exception as error:
        print(f"{type(error).__name__:<16} {error}")

### **Reading a traceback**

When an exception is not handled, Python prints a **traceback**. Read it **from the bottom up**:

```text
Traceback (most recent call last):
  File "main.py", line 12, in <module>
    report = build_report(data)                   ← 3. where it started
  File "main.py", line 8, in build_report
    average = total / count                       ← 2. the line that failed
ZeroDivisionError: division by zero               ← 1. WHAT happened (start here)
```

1. The **last line** names the exception class and its message.
2. The lines above show the **call chain** that led there, most recent last.

## **`try`, `except`, `else`, and `finally`**

```python
try:
    # code that might raise an exception
except SomeError:
    # runs only if SomeError was raised
else:
    # runs only if NO exception was raised
finally:
    # ALWAYS runs — use for cleanup
```

```{mermaid}
flowchart TD
    T["try block"] -->|exception| X["matching except block"]
    T -->|no exception| E["else block"]
    X --> F["finally block"]
    E --> F
    F --> N["continue program"]
```

In [ ]:
def parse_score(text):
    try:
        score = float(text)
    except ValueError:
        print(f"  '{text}' is not a number")
        return None
    else:
        print(f"  parsed {score}")
        return score
    finally:
        print("  (finished checking)")


for raw in ["92.5", "ninety"]:
    print(f"input: {raw}")
    parse_score(raw)

### **Handling several exception types**

Catch **specific** exceptions, from most specific to most general. Use `as` to access the exception **object**:

In [ ]:
def safe_average(scores):
    try:
        return sum(scores) / len(scores)
    except ZeroDivisionError:
        return 0.0
    except TypeError as error:
        print("Bad data:", error)
        return None


print(safe_average([90, 80]))
print(safe_average([]))
print(safe_average([90, "80"]))

## **Exceptions Are Objects in a Class Hierarchy**

Every exception is an instance of a class that inherits from `BaseException`. An `except` clause catches the named class **and all of its subclasses** — inheritance at work.

```{mermaid}
classDiagram
BaseException <|-- Exception
BaseException <|-- KeyboardInterrupt
Exception <|-- ArithmeticError
Exception <|-- LookupError
Exception <|-- ValueError
Exception <|-- TypeError
ArithmeticError <|-- ZeroDivisionError
LookupError <|-- IndexError
LookupError <|-- KeyError
```

In [ ]:
print(ZeroDivisionError.__mro__)

try:
    {"a": 1}["b"]
except LookupError as error:         # catches KeyError AND IndexError
    print("Caught", type(error).__name__, "via LookupError")
    print("args:", error.args)

## **Raising Exceptions**

Classes should **raise** exceptions when they are asked to do something invalid. Use `raise` with an exception **instance** and a clear message:

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        if balance < 0:
            raise ValueError("Initial balance cannot be negative.")
        self.owner = owner
        self._balance = balance

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f"Withdrawal must be positive, got {amount}.")
        if amount > self._balance:
            raise ValueError(f"Insufficient funds: balance is {self._balance}.")
        self._balance -= amount
        return self._balance


acct = BankAccount("Ana", 1000)
for amount in (300, -50, 5000):
    try:
        print("New balance:", acct.withdraw(amount))
    except ValueError as error:
        print("Rejected:", error)

Choose the **built-in exception that matches the problem**:

| Situation                                     | Exception            |
| --------------------------------------------- | -------------------- |
| right type, but invalid value                 | `ValueError`         |
| wrong type of argument                        | `TypeError`          |
| missing key / index                           | `KeyError` / `IndexError` |
| operation not allowed in the current state    | `RuntimeError` (or a custom exception) |
| a method a subclass must provide              | `NotImplementedError` |
| a file that does not exist                    | `FileNotFoundError`  |

## **Defining Custom Exceptions**

For a real application, built-in exceptions are often too general. **Custom exception classes** describe problems in the language of your **domain** and let callers handle them precisely.

A good practice is to create **one base exception** for your package, then specific subclasses:

```{mermaid}
classDiagram
Exception <|-- GradebookError
GradebookError <|-- InvalidScoreError
GradebookError <|-- StudentNotFoundError
GradebookError <|-- DuplicateStudentError
class InvalidScoreError {
    +float score
    +float low
    +float high
}
class StudentNotFoundError {
    +String student_id
}
```

In [ ]:
class GradebookError(Exception):
    '''Base class for all gradebook errors.'''


class InvalidScoreError(GradebookError):
    def __init__(self, score, low=0, high=100):
        self.score = score
        self.low = low
        self.high = high
        super().__init__(f"Score {score} is outside the range {low}–{high}.")


class StudentNotFoundError(GradebookError, KeyError):
    def __init__(self, student_id):
        self.student_id = student_id
        super().__init__(f"No student with ID {student_id!r}.")

    def __str__(self):                  # KeyError would otherwise add quotes
        return self.args[0]


class DuplicateStudentError(GradebookError):
    pass


class Gradebook:
    def __init__(self):
        self._scores = {}

    def add_student(self, student_id):
        if student_id in self._scores:
            raise DuplicateStudentError(f"{student_id} is already enrolled.")
        self._scores[student_id] = []

    def record(self, student_id, score):
        if student_id not in self._scores:
            raise StudentNotFoundError(student_id)
        if not 0 <= score <= 100:
            raise InvalidScoreError(score)
        self._scores[student_id].append(score)

In [ ]:
book = Gradebook()
book.add_student("2026-001")

attempts = [("2026-001", 95), ("2026-001", 120), ("2026-999", 80)]
for sid, score in attempts:
    try:
        book.record(sid, score)
        print(f"Recorded {score} for {sid}")
    except InvalidScoreError as error:
        print("Invalid score:", error, "| offending value:", error.score)
    except StudentNotFoundError as error:
        print("Unknown student:", error)

# callers who don't care about the details can catch the base class
try:
    book.add_student("2026-001")
except GradebookError as error:
    print(f"{type(error).__name__}: {error}")

Custom exceptions can carry **attributes** (`error.score`) that help the caller decide what to do — they are full objects, not just messages.

### **Exception chaining**

When you catch a low-level exception and raise a domain exception, use `raise ... from ...` to keep the original cause visible for debugging:

In [ ]:
class ConfigError(Exception):
    pass


def load_threshold(settings):
    try:
        return float(settings["threshold"])
    except (KeyError, ValueError) as error:
        raise ConfigError("Invalid or missing 'threshold' setting.") from error


try:
    load_threshold({"threshold": "high"})
except ConfigError as error:
    print("Error:", error)
    print("Caused by:", repr(error.__cause__))

## **Defensive Programming and Validation**

**Defensive programming** means writing classes that **protect themselves** from invalid use. Combine it with encapsulation: validate data at the **boundary** where it enters the object.

### **Validate in constructors and setters**

In [ ]:
class Sensor:
    def __init__(self, sensor_id, low, high):
        if not isinstance(sensor_id, str) or not sensor_id:
            raise TypeError("sensor_id must be a non-empty string.")
        if low >= high:
            raise ValueError(f"low ({low}) must be less than high ({high}).")
        self._sensor_id = sensor_id
        self._low, self._high = low, high
        self._readings = []

    def record(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"Reading must be numeric, got {type(value).__name__}.")
        if not self._low <= value <= self._high:
            raise ValueError(f"{value} outside [{self._low}, {self._high}].")
        self._readings.append(value)


for args in [("T1", 0, 50), ("", 0, 50), ("T2", 50, 0)]:
    try:
        Sensor(*args)
        print("Created", args)
    except (TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

### **EAFP vs. LBYL**

Python programmers often prefer **EAFP** — *"easier to ask forgiveness than permission"* — over **LBYL** — *"look before you leap"*.

```python
# LBYL: check first
if "age" in record:
    age = record["age"]
else:
    age = None

# EAFP: try, and handle the failure
try:
    age = record["age"]
except KeyError:
    age = None
```

EAFP is usually clearer when failure is **rare**, and it avoids race conditions (for example, a file deleted between the check and the open). LBYL is fine when the check is cheap and makes the intent obvious.

### **`assert` is not validation**

`assert condition, "message"` is for **catching programmer mistakes during development**. Assertions can be **disabled** (`python -O`), so never use them to validate user input or data — raise a proper exception instead.

### **Cleanup with `finally` and context managers**

Resources must be released even when errors occur. `finally` guarantees it, and the `with` statement (which uses `__exit__()`) does it automatically:

In [ ]:
class Connection:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"open {self.name}")
        return self

    def __exit__(self, exc_type, exc, tb):
        print(f"close {self.name} (error: {exc_type.__name__ if exc_type else None})")
        return False                       # re-raise any exception

    def query(self, sql):
        if "DROP" in sql:
            raise PermissionError("Destructive queries are not allowed.")
        return ["row1", "row2"]


try:
    with Connection("grades.db") as conn:
        print(conn.query("SELECT * FROM grades"))
        conn.query("DROP TABLE grades")
except PermissionError as error:
    print("Handled:", error)

## **Common Mistakes**

**Bare `except:`** catches *everything* — including `KeyboardInterrupt` and your own bugs — and hides them:

```python
try:
    process(data)
except:              # ✗ never do this
    pass
```

**Swallowing exceptions.** Catching an error and doing nothing makes bugs invisible. At minimum, log or report it.

**Catching too broadly too early.** Handle an exception **where you can do something useful about it**. Low-level classes should usually *raise*; the top level of the application (the UI or main script) decides how to *report*.

**Using exceptions for normal control flow.** Exceptions are for *exceptional* situations, not for ordinary branching such as "end of menu loop".

**Vague messages.** `raise ValueError("Error")` helps no one. Say *what* was wrong and *what was expected*: `"Score 120 is outside the range 0–100."`

## **Summary**

| Concept                    | Key idea                                                                |
| -------------------------- | ----------------------------------------------------------------------- |
| `try` / `except`           | handle specific exceptions where you can respond meaningfully           |
| `else` / `finally`         | code for the success path / code that always runs                        |
| exception hierarchy        | exceptions are classes; `except Parent` also catches subclasses          |
| `raise`                    | signal invalid use with a precise exception and a clear message          |
| custom exceptions          | one base class per package, specific subclasses with useful attributes   |
| `raise ... from ...`       | translate low-level errors while keeping the original cause              |
| defensive programming      | validate at object boundaries; fail fast; release resources reliably     |

## **References**

Downey, A. B. (2024). *Think Python: How to think like a computer scientist* (3rd ed.). O'Reilly Media. https://allendowney.github.io/ThinkPython/

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Python Software Foundation. (2026). *Built-in exceptions*. https://docs.python.org/3/library/exceptions.html

Python Software Foundation. (2026). *The Python language reference: 3. Data model*. https://docs.python.org/3/reference/datamodel.html

Python Software Foundation. (2026). *The Python tutorial: 8. Errors and exceptions*. https://docs.python.org/3/tutorial/errors.html

Ramalho, L. (2022). *Fluent Python: Clear, concise, and effective programming* (2nd ed.). O'Reilly Media.

Thomas, D., & Hunt, A. (2019). *The pragmatic programmer: Your journey to mastery* (20th anniversary ed.). Addison-Wesley.